<a href="https://colab.research.google.com/github/sujeet-banerjee/colab-genai-scratch/blob/sujeet-banerjee-patch-1/RAG_load_raw_docs_videos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load a doc from the Google Drive

# Load docs from various sources


In [5]:
! pip install --upgrade pip
! pip install openai
! pip install pypdf
!pip install langchain_community

!pip install langchain
!pip install langgraph
!pip install langchain-core
!pip install langchain-openai
!pip install --upgrade langchain

Load Open AI Client

In [6]:
from openai import OpenAI

# For secrets
from google.colab import userdata

open_api_key = userdata.get('open_api_key')
c_key = userdata.get('c_key')
hf_sujeet = userdata.get('hf_sujeet')

client = OpenAI(
  api_key= open_api_key
)

Load PDF/file from Google Drive!

In [7]:
from google.colab import drive
drive.mount('/content/drive')

!ls /content/drive/MyDrive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
'A B Test.gform'
'A B Test (Responses).gsheet'
 APAL_Study
 Blue_laptop_backup
 Code
'Colab Notebooks'
'Customer Feedback.gform'
'Docker Swarm with Mixed OS (Windows and Ubuntu).gdoc'
 Dot_product.drawio
 Emulator_No_Begin.png
 Emulator_No_singnin.png
 GenAI_Posts
'Github Token No Expiry.gdoc'
'give example python code for PEFT based on adapte...'
'Hands-On Generative AI Techniques: A Guided Journey.gdoc'
'how meta AI seach is going to dent Google?.gdoc'
 icons
'India Employment Agreement_SIGNED_AND_ACCEPTED.pdf'
'India PIIA.pdf'
 Jobs
'Leetcode for RateLimit UniqueNameGen and GroupMembership.gdoc'
'Open-source agentic AI platforms.'$'\n'' (1).gdoc'
'Open-source agentic AI platforms.'$'\n''.gdoc'
'Permissions for '\''suzlab.pem'\'' are too open. how to set pem perms using powershell?.gdoc'
'Prep Doc for Coding & Design.pdf'
'prove that 2^n > n!.gdoc'
'reverse

# Spit or Do chunking

In [14]:
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter, TokenTextSplitter

#Test chunking
chunk_size =120
chunk_overlap = 24

text_splitter = TokenTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)
c_splitter = CharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separator = ' '
)





In [15]:
texts = text_splitter.split_documents(pages)
print(len(texts))
texts[0].page_content[0:20]

905


'Human Resource Analy'

In [16]:
r_splitter.split_text(texts[0].page_content)

['Human Resource Analytics \n(Why are our employees leaving?) \n \nBackground',
 'Background \nHari Prasad, the senior manager (HR) of Chaurasia and Company was going through the employee',
 'records of his organization. The most startling fact that caught his eye were the names Amar Akbar',
 'and Anthony. All the three were top performing and experienced employees with whom he had',
 'recently interacted in an annual meeting. But his anxiety became worse as he noticed that this was',
 'a pattern- some of the experienced and valuable employees had recently left the company']

In [17]:
text_splitter.split_text(texts[0].page_content)

['Human Resource Analytics \n(Why are our employees leaving?) \n \nBackground \nHari Prasad, the senior manager (HR) of Chaurasia and Company was going through the employee \nrecords of his organization. The most startling fact that caught his eye were the names Amar Akbar \nand Anthony. All the three were top performing and experienced employees with whom he had \nrecently interacted in an annual meeting. But his anxiety became worse as he noticed that this was \na pattern- some of the experienced and valuable employees had recently left the company']

In [18]:
c_splitter.split_text(texts[0].page_content)

['Human Resource Analytics \n(Why are our employees leaving?) \n \nBackground \nHari Prasad, the senior manager (HR) of',
 'senior manager (HR) of Chaurasia and Company was going through the employee \nrecords of his organization. The most',
 'organization. The most startling fact that caught his eye were the names Amar Akbar \nand Anthony. All the three were top',
 'All the three were top performing and experienced employees with whom he had \nrecently interacted in an annual meeting.',
 'in an annual meeting. But his anxiety became worse as he noticed that this was \na pattern- some of the experienced and',
 'of the experienced and valuable employees had recently left the company']

### Context aware splitting

Context aware splitting
Chunking aims to keep text with common context together.

A text splitting often uses sentences or other delimiters to keep related text together but many documents (such as Markdown) have structure (headers) that can be explicitly used in splitting.

We can use MarkdownHeaderTextSplitter to preserve header metadata in our chunks, as show below.

In [19]:
# prompt: MarkdownHeaderTextSplitter

from langchain.text_splitter import MarkdownHeaderTextSplitter

markdown_document = """
# Title

## Header 1
This is some text under Header 1.

### Sub-header 1.1
This is some text under Sub-header 1.1.

## Header 2
This is some text under Header 2.
"""

headers_to_split_on = [
    ("#", "TitleSuz"),
    ("##", "HeaderSuz"),
    ("###", "Sub-header-suz"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

md_header_splits = markdown_splitter.split_text(markdown_document)

for split in md_header_splits:
    print(split.metadata)
    print("---")

{'TitleSuz': 'Title', 'HeaderSuz': 'Header 1'}
---
{'TitleSuz': 'Title', 'HeaderSuz': 'Header 1', 'Sub-header-suz': 'Sub-header 1.1'}
---
{'TitleSuz': 'Title', 'HeaderSuz': 'Header 2'}
---


# Embeddings and Vector-DB storage

#### RecursiveCharacterTextSplitter

In [20]:
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
    separators=["\n\n", "\n", "(?<=\. )", " ", ""]
)

#Cunked docs
docs = r_splitter.split_documents(pages);
print('Pages len: ', len(pages))
print('Docs len: ', len(docs))

print(docs[0:5])


Pages len:  220
Docs len:  355
[Document(metadata={'producer': 'Adobe PDF Library 11.0', 'creator': 'Acrobat PDFMaker 11 for Word', 'creationdate': '2024-04-14T17:42:26+05:30', 'author': 'IIMT', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2024-04-14T17:42:28+05:30', 'sourcemodified': 'D:20240414085852', 'subject': '', 'title': '', 'source': '/content/drive/MyDrive/APAL_Study/Case_HRA_IIMC.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Human Resource Analytics \n(Why are our employees leaving?) \n \nBackground \nHari Prasad, the senior manager (HR) of Chaurasia and Company was going through the employee \nrecords of his organization. The most startling fact that caught his eye were the names Amar Akbar \nand Anthony. All the three were top performing and experienced employees with whom he had \nrecently interacted in an annual meeting. But his anxiety became worse as he noticed that this was \na pattern- some of the experienced and valuable employees

#### Create Embeddings using OpenAI API

In [21]:
from langchain.embeddings import OpenAIEmbeddings as oaE
# Alternative
from langchain.embeddings import HuggingFaceEmbeddings as hfE
import numpy as np

#print(open_api_key)

In [22]:
#Using Open API Embeddings
'''
dot of vec1 and vec2 0.9630313341962432
dot of vec1 and vec3 0.7701484645728846
'''
#

s1 = "i like dogs"
s2 = "i like canines"
s3 = "the weather is ugly outside"

vec1 = oaE(openai_api_key=open_api_key).embed_query(text=s1)
vec2 = oaE(openai_api_key=open_api_key).embed_query(text=s2)
vec3 = oaE(openai_api_key=open_api_key).embed_query(text=s3)

print(vec1)
print(vec2)
print(vec3)
print(np.linalg.norm(vec1))
print(np.linalg.norm(vec2))
print(np.linalg.norm(vec3))
print("dot of vec1 and vec2", np.dot(vec1, vec2))
print("dot of vec1 and vec3", np.dot(vec1, vec3))

print("Vector length OpenAI: ", len(vec1), "")

/tmp/ipython-input-877374633.py:12: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  vec1 = oaE(openai_api_key=open_api_key).embed_query(text=s1)


[-0.027582872389708395, -0.005395962237097065, -0.025852627683023665, -0.03306408417492009, -0.027406058926058477, 0.02266998906261565, -0.010400409479725098, -0.008057633880331566, 0.0025369547376035404, -0.019664162043212297, 0.0006642337922942595, 0.02922470850192291, -0.0054401656030095445, 0.0006748899858085883, 0.00015037021653449518, 0.014081914791334329, 0.0299067030241947, -0.0011942790114112447, 0.004205629938154387, -0.0039056790920513, -0.011720194227033572, 0.006889403264345128, 0.013033664606921389, -0.04723440383543412, -0.002336461231972875, 0.004679237297027721, 0.01694881641425696, -0.00012747922309875727, -0.02589051575019286, -0.016342598026323558, 0.026724064403786687, 0.002964779905878574, -0.0155216799702163, -0.024741230099113647, 0.006071641460625571, -0.014978610844425233, 0.009718415888775936, -0.011436029997974096, -0.0043729713850120065, -0.010855072805013834, -0.01711299928042031, 0.011164495435078563, 0.006939920997678264, -0.02201325387267174, -0.0026143

In [23]:
## Using HuggingFace Embeddings
# Result
'''
dot of vec1 and vec2 0.8981182842559048
dot of vec1 and vec3 0.03641026493961007
'''
#

s1 = "i like dogs"
s2 = "i like canines"
s3 = "the weather is ugly outside"

vec1 = hfE().embed_query(s1)
vec2 = hfE().embed_query(s2)
vec3 = hfE().embed_query(s3)

print(vec1)
print(vec2)
print(vec3)
print(np.linalg.norm(vec1))
print(np.linalg.norm(vec2))
print(np.linalg.norm(vec3))
print("dot of vec1 and vec2", ( np.dot(vec1, vec2)/(np.linalg.norm(vec1) * np.linalg.norm(vec2)) ))
print("dot of vec1 and vec3", ( np.dot(vec1, vec3)/(np.linalg.norm(vec1) * np.linalg.norm(vec3)) ) )

print("Vector length HuggingFace: ", len(vec1))

/tmp/ipython-input-3719861435.py:13: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  vec1 = hfE().embed_query(s1)
/tmp/ipython-input-3719861435.py:13: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  vec1 = hfE().embed_query(s1)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipython-input-3719861435.py:14: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  vec2 = hfE().embed_query(s2)
/tmp/ipython-input-3719861435.py:15: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  vec3 = hfE().embed_query(s3)


[-0.013610822148621082, 0.11503763496875763, -0.027678335085511208, -0.041481710970401764, 0.04854816943407059, 0.03823169320821762, -0.045007575303316116, -0.012027067132294178, -0.00249747047200799, -0.017130006104707718, -0.08121422678232193, 0.017644111067056656, -0.05390755832195282, -0.013921351172029972, -0.017844628542661667, -0.011630043387413025, 0.03812805190682411, 0.08376377075910568, 0.016784565523266792, -0.010697281919419765, 3.216173354303464e-05, 0.030750378966331482, -0.05290459841489792, -0.03610792011022568, 0.023436011746525764, 0.03191141411662102, -0.025538088753819466, -0.0515754334628582, 0.035590071231126785, 0.03687377646565437, -0.05278579145669937, -0.029876425862312317, -0.012499134987592697, -0.02516518533229828, 1.4985516827437095e-06, 0.01840418577194214, -0.03838668018579483, 0.020760133862495422, 0.060190360993146896, -0.0023674792610108852, -0.03244411200284958, -0.06694810837507248, -0.038730520755052567, -0.05493103712797165, -0.03285567834973335,

#### Store in Chroma DB (Vector Store)

In [24]:
! pip install chromadb
! pip install tiktoken

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 118.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 93.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 139.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 108.9 MB/s  0:00:00
  Created wheel for pypika: filename=pypika-0.48.9-py2.py3-none-any.whl size=53803 sha256=f75a31513d7d409eac8f34e24d88669225dbb6d9bc66f4709d919e5870e6a65a
  Stored in directory: /root/.cache/pip/wheels/a3/01/bd/4c40ceb9d5354160cb186dcc153360f4ab7eb23e2b24daf96d
Successfully built pypika
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22/22 [chromadb]


##### Using HuggingFace for Apriel (Service Now)

@Read:
https://huggingface.co/ServiceNow-AI/Apriel-5B-Base

To load chromaDB from a handful of texts:

texts = [
    """The Amanita phalloides has a large and imposing epigeous (aboveground) fruiting body (basidiocarp).""",

    """A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all-white.""",
    
    """A. phalloides, a.k.a Death Cap, is one of the most poisonous of all known mushrooms.""",
]

smalldb = Chroma.from_texts(texts, embedding=embedding)

In [ ]:
from langchain.vectorstores import Chroma

persist_directory = '/content/chroma_db/'
!rm -rf persist_directory  # remove old database files if any
# Change persist_directory to a location within the Colab environment's writable file system

# Does not work!
#embedding=hfE(model_name="ServiceNow/msrc-attendee-apriel-small-v2", trust_remote_code=True),

vectordb = Chroma.from_documents(
    documents=docs,
    embedding=hfE(),
    persist_directory=persist_directory
)
print(vectordb._collection.count())

/tmp/ipython-input-418028310.py:12: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding=hfE(),


##### Persist DB And Query

In [ ]:
vectordb.persist()


In [ ]:
question = "is there an email i can ask for help"
qe = vectordb.similarity_search(question,k=3)
print("Searched docs: ", len(qe))
print("\n\n-----\n".join([x.page_content for x in qe]))


In [ ]:
question = "what did they say about the supply chain in the chapter Impacts of technology disruption?"
qe = vectordb.similarity_search(question,k=3)
print("Searched docs: ", len(qe))
print("\n\n-----\n".join([x.page_content for x in qe]))

##### Problems with the above vector search
1. There is no diversification of the responses. Sometimes there are a repeat of the chunks, adding no value to the final prompting to the LLM
2. Often the most relevant chunk might be at teh bottom (or out) of the reported list, esp with specified filters.

Solution:
1. Using diversification of the reported docs - *MMR*
2. filter the obvious records (say, chapter name etc), by specifying the query-filters passed on to the db search

In [ ]:
question = "is there an email i can ask for help"
qe = vectordb.max_marginal_relevance_search(question, k=3, fetch_k=10)
print("Searched docs: ", len(qe))
print("\n\n-----\n".join([x.page_content for x in qe]))

# Retrieval

In [ ]:
!pip install lark

#### MMR - Max Marginal Retrieval
We saw the example above using
> vectordb.max_marginal_relevance_search(question, k=3, fetch_k=10)

MMR brings more diversified results as opposed to the very high ranked similarity matches.



#### Specify DB filter

In [ ]:
question = "what did they say about the supply chain in the chapter Impacts of technology disruption?"
qe = vectordb.similarity_search(
    question,
    k=3,
    filter={"source":"/content/drive/MyDrive/APAL_Study/DUP_Digital-supply-network.pdf"}
    )
print("Searched docs: ", len(qe))
print("\n\n-----\n".join([x.page_content for x in qe]))

#### Self Query Retriever



Addressing Specificity: working with metadata using self-query retriever
But we have an interesting challenge: we often want to infer the metadata from the query itself.

To address this, we can use SelfQueryRetriever, which uses an LLM to extract:

The query string to use for vector search
A metadata filter to pass in as well
Most vector databases support metadata filters, so this doesn't require any new databases or indexes.

In [ ]:
#Check the vector DB count
print(vectordb._collection.count())

In [ ]:
from langchain.llms import OpenAI
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo

metadata_field_info = [
    AttributeInfo(
        name="source",
        description="The document the chunk is from, for example 'DUP_Digital-supply-network.pdf'",
        type="string",
    ),
    AttributeInfo(
        name="page",
        description="The page or paragraph from the document",
        type="integer",
    ),
    AttributeInfo(
        name="chapter",
        description="The chapter from the document, containing multiple pages",
        type="integer",
    ),
]

document_content_description = "Documents"
llm = OpenAI(
    openai_api_key = open_api_key,
    model='gpt-3.5-turbo-instruct',
    temperature=0,)
retriever = SelfQueryRetriever.from_llm(
    llm,
    vectordb,
    document_content_description,
    metadata_field_info,
    verbose=True
)

In [ ]:
question = "what did they say about supply chain in the chapter Impacts of technology disruption?"
docs = retriever.get_relevant_documents(question)
for d in docs:
    print(d.metadata)

In [ ]:
len(docs)

#### Contextual Compression Retriever and LLMChainExtractor

Often the RAG retrieves the entire set of docs, while only a section or a few lines from the retrieved docs are relevant to the context/question. Contextual Compression (using LLM) can help narrow down the response to only the relevant text.

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

def pretty_print(docs):
  print("\n\n".join([x.page_content for x in docs]))

prompt = f"""
Hello LLM. I am using you as doc retriever, and a contextual compressor.

Pick as many relevant documents as possible, and as diverse as possible
""".strip()

### DOES NOT WORK WITH ARG prompt
# Error:
'''
/usr/local/lib/python3.11/dist-packages/langchain_community/llms/openai.py in _generate(self, prompts, stop, run_manager, **kwargs)
    461                 )
    462             else:
--> 463                 response = completion_with_retry(
    464                     self, prompt=_prompts, run_manager=run_manager, **params
    465                 )

TypeError: langchain_community.llms.openai.completion_with_retry() got multiple values for keyword argument 'prompt'
'''

#llm = OpenAI(openai_api_key=open_api_key, temperature=0, prompt=prompt)
#print(llm.get_num_tokens_from_messages(messages=prompt))

llm = OpenAI(openai_api_key=open_api_key, temperature=0)

print(client.get_api_list)
#print(llm.get_api_list)

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectordb.as_retriever()
)

# Asking the same question
'''
"what did they say about supply chain in the chapter Impacts of technology disruption?"
'''
compressed_docs = compression_retriever.get_relevant_documents(
    query = question,
    #prompt = prompt
    )

pretty_print(compressed_docs)


# Chat Bot

## Map Reduce

## Refine

## Re-Rank